# Edge AI Assistant: Architecture & Deployment Tutorial

## What You'll Learn

This tutorial teaches edge AI deployment using a production-ready system with:

1. **Orchestrator Agent Pattern** - How a main agent coordinates specialized sub-agents
2. **Dynamic Model Selection** - Task complexity analysis determines local vs cloud model usage
3. **Multi-Modal Processing** - Voice, text, and image processing across different deployment modes
4. **Resource-Aware Configuration** - How the system adapts resource limits to automotive, IoT, and development environments

## System Architecture Overview

```mermaid
graph TD
    A[User Input] --> B[Main Orchestrator]
    B --> C{Intent Classification}
    C -->|Calendar| D[Calendar Agent]
    C -->|Vehicle| E[Vehicle Agent]
    C -->|Search| F[Search Agent]
    C -->|Voice| G[Voice Processing]
    
    D --> H[Appointment Tools]
    E --> I[FAISS Knowledge Base]
    F --> J[Web Search Tools]
    G --> K[Qwen2.5-Omni Model]
    
    H --> L[Response]
    I --> L
    J --> L
    K --> L
```

## Why This Architecture?

- **Specialization**: Each agent excels at specific domain tasks
- **Scalability**: Easy to add new agents without changing existing ones  
- **Resource Efficiency**: Agents can use different models based on task complexity
- **Deployment Flexibility**: Same code runs in development, containers, and edge devices

## Dynamic Model Selection Flow

```mermaid
flowchart TD
    A[User Query] --> B[Local Analyzer]
    B --> C{Task Complexity}
    C -->|Simple| D[LlamaCpp Local]
    C -->|Complex| E[Bedrock Cloud]
    D --> F[Response]
    E --> F
```

**Examples by Complexity:**
- **Simple**: "What time is it?", "Schedule a meeting", "Hello"
- **Complex**: Multi-step analysis, creative writing, detailed explanations

**Key Insight**: A local LlamaCpp model analyzes each query first to determine if it needs local or cloud processing.

## Step 1: Build and Deploy the Complete System

**What happens during build:**
1. **Container Creation**: Multi-architecture Docker container (x86_64/ARM64)
2. **Model Integration**: Downloads Qwen2.5-Omni (voice/vision/text) model
3. **Agent Packaging**: Bundles orchestrator + specialized agents  
4. **Knowledge Base**: Includes FAISS vector store with vehicle diagnostics
5. **API Server**: FastAPI endpoint for programmatic access
6. **Resource Configuration**: Sets up automotive-optimized memory limits

In [ ]:
# Build and start the edge AI container (takes 5-10 minutes first time)
import subprocess
import os

os.chdir('src/edge')
subprocess.run(['./deployment/setup.sh'])

## Step 2: Test REST API

Test the container's REST API endpoint to ensure it's running and responsive. This uses the same API that would be integrated into automotive systems or mobile apps.

In [ ]:
import requests
import time

# Wait for container startup
time.sleep(30)

# Test basic chat
response = requests.post("http://localhost:8000/chat", json={
    "prompt": "Hello, what can you help me with?",
    "session_id": "test_session"
})

print(response.json()['response'])

## Step 3: Voice Processing

Record audio from your microphone, encode it, and send to the container for speech-to-text processing using Qwen2.5-Omni. This demonstrates the complete voice pipeline used in automotive and IoT deployments.

## Step 3: Multi-Modal Voice Processing

**Voice Pipeline Sequence:**

```mermaid
sequenceDiagram
    participant User
    participant Audio
    participant Model
    participant Orchestrator
    participant Agent
    
    User->>Audio: Record voice input
    Audio->>Model: Send audio data
    Model->>Orchestrator: Transcribed text
    Orchestrator->>Agent: Route to specialist
    Agent->>Orchestrator: Generated response
    Orchestrator->>User: Final answer
```

**Deployment-Specific Voice Handling:**
- **Development**: Direct microphone access
- **Container**: File-based audio exchange via volume mounts  
- **API**: Base64-encoded audio in JSON payload

This demonstrates how the same orchestrator handles different input modalities while maintaining the same agent coordination logic.

In [ ]:
import subprocess
import base64
import os

# Record 10 seconds of audio
os.makedirs("../../audio_exchange", exist_ok=True)
subprocess.run([
    "python", "-m", "src.utils.audio_cli", "record", 
    "--duration", "10", "--output", "../../audio_exchange/voice_input.wav"
], cwd="../..")

# Send to container
with open("../../audio_exchange/voice_input.wav", "rb") as f:
    audio_data = base64.b64encode(f.read()).decode('utf-8')

response = requests.post("http://localhost:8000/chat", json={
    "prompt": "voice",
    "audio_data": audio_data,
    "audio_format": "wav",
    "session_id": "voice_session"
})

result = response.json()
print(f"You said: {result.get('transcription', 'N/A')}")
print(f"Response: {result['response']}")

In [ ]:
# Test natural language scheduling
appointments = [
    "Schedule a dentist appointment tomorrow at 2 PM",
    "Block time for project review every Friday at 10 AM", 
    "Set up lunch meeting with Sarah next Tuesday at 12:30"
]

for request in appointments:
    response = requests.post("http://localhost:8000/chat", json={
        "prompt": request,
        "session_id": "calendar_session"
    })
    print(f"✓ {request}")
    print(response.json()['response'][:150] + "...\n")

In [ ]:
# View created schedule
response = requests.post("http://localhost:8000/chat", json={
    "prompt": "Show me my appointments for this week",
    "session_id": "calendar_session"
})

print(response.json()['response'])

## Step 5: Knowledge Retrieval - Vehicle Diagnostic Agent

**FAISS Vector Search Architecture:**

```mermaid
flowchart TB
    A[User Query] --> B[Vector Embedding]
    B --> C[FAISS Index Search]
    C --> D[Similarity Matching]
    D --> E[Top K Documents]
    E --> F[LLM Synthesis]
    F --> G[Response with Sources]
    
    H[(Vehicle Manuals)] --> I[Pre-processed Vectors]
    I --> C
    J[(Maintenance Data)] --> I
    K[(Diagnostic Codes)] --> I
```

**Offline Knowledge System:**
1. **Pre-indexed Knowledge**: Vehicle manuals, diagnostics, maintenance schedules stored as vectors
2. **Semantic Search**: FAISS finds relevant information using vector similarity  
3. **Local Operation**: No internet required - critical for automotive use
4. **Source Attribution**: Responses include manual sections and page numbers

In [ ]:
# Test diagnostic scenarios
questions = [
    "My tire pressure warning light came on. What should I do?",
    "How often should I change my oil?",
    "The car makes a grinding noise when I brake",
    "Check engine light is on and car runs rough"
]

for question in questions:
    response = requests.post("http://localhost:8000/chat", json={
        "prompt": question,
        "session_id": "vehicle_session"
    })
    
    print(f"Q: {question}")
    print(f"A: {response.json()['response'][:200]}...\n")

## Step 6: Monitor Performance

Check resource usage and response latency to understand edge computing constraints. Important for optimizing deployment to resource-limited devices.

In [ ]:
# Container resource usage
subprocess.run([
    "docker", "stats", "strands-edge-personal-assistant", 
    "--no-stream", "--format", "table {{.Container}}\t{{.CPUPerc}}\t{{.MemUsage}}"
], cwd="../..")

In [ ]:
# Response latency test
start_time = time.time()
response = requests.post("http://localhost:8000/chat", json={
    "prompt": "What time is it?",
    "session_id": "perf_test"
})
latency = (time.time() - start_time) * 1000

print(f"Response latency: {latency:.0f}ms")
print(f"Response: {response.json()['response']}")

## Step 7: Interactive Chat

Connect directly to the container for conversational interaction. This shows how the system would work in kiosk or terminal applications.

In [ ]:
# Container management commands
commands = {
    "Interactive chat": "docker exec -it strands-edge-personal-assistant python /app/main.py",
    "View logs": "./deployment/setup.sh logs",
    "Stop container": "./deployment/setup.sh stop", 
    "Restart": "./deployment/setup.sh restart"
}

for desc, cmd in commands.items():
    print(f"{desc:15}: {cmd}")

In [ ]:
# Start interactive session (run this in terminal for best experience)
# subprocess.run(["docker", "exec", "-it", "strands-edge-personal-assistant", "python", "/app/main.py"], cwd="../..")
print("Run the above command in terminal for interactive chat")

# Production deployment examples

In [ ]:
# Production deployment examples
automotive = """docker run -d --name automotive-assistant --restart always \\
  -p 8000:8000 -e DEPLOYMENT_TARGET=automotive -e ENABLE_API=true \\
  --memory=4g --cpus=2.0 personal-assistant:edge"""

print(automotive)

## Architecture Mastery - What You've Learned

### Orchestrator Agent Pattern
- **Central Coordinator**: One main agent (`main.py`) routes to specialized sub-agents
- **Intent Classification**: Smart routing based on domain detection  
- **Unified Interface**: Single entry point for multiple capabilities

### Task-Based Model Selection  
- **Complexity Analysis**: Local LlamaCpp model analyzes each query to determine complexity
- **Smart Routing**: Simple tasks use local model, complex tasks use cloud model
- **Environment Independent**: Model choice based on task needs, not deployment environment

### Multi-Modal Processing
- **Voice Pipeline**: Qwen2.5-Omni handles speech-to-text across all deployment modes
- **Deployment Adaptation**: Different audio handling per environment (direct/file/API)
- **Consistent Logic**: Same orchestrator regardless of input modality

### Specialized Agents
- **Domain Expertise**: Each agent optimized for specific tasks (calendar/vehicle/search)
- **Dedicated Tools**: Custom function calling per domain (`create_appointment`, `search_vehicle_knowledge`)
- **Model Flexibility**: Each agent can use the model best suited for its complexity needs

### Resource-Aware Configuration
- **Environment Constraints**: Different memory, context window, and token limits per deployment
- **Performance Optimization**: Automotive (4GB), IoT (6GB), development (8GB) configurations
- **UI Adaptation**: Rich interfaces for development, simple for resource-constrained environments

### Key Architectural Insight
This **orchestrator pattern** with **task-based model selection** enables **one unified system** to handle **multiple specialized domains** while **optimizing resource usage** based on both **task complexity** and **deployment constraints**.